# AI Chatbot With Memory

Cette notebook fournit un squelette de travail pour construire un chatbot local à mémoire courte en utilisant **LangChain**, **llama.cpp** et **Streamlit**. L'objectif est de créer un assistant conversationnel personnalisé capable d'utiliser un modèle quantifié (comme **Mistral 7B** en version `.gguf`) pour répondre de façon contextuelle en conservant l'historique des échanges.

🚨 **Remarque :** L'environnement ici ne dispose pas des librairies `llama-cpp-python`, `langchain` ou `streamlit`. Utilisez ce notebook comme guide en le chargeant dans Google Colab ou votre machine locale avec un GPU et installez les dépendances via `pip` comme indiqué ci‑dessous.


## 1. Préparation de l'environnement Python

Avant de commencer, créez un répertoire de projet (par exemple `ai_chatbot_memory`) et un environnement virtuel dédié. Installez les dépendances nécessaires :

```bash
# Depuis un terminal à la racine du projet
python3 -m venv .venv
source .venv/bin/activate  # sous Windows utilisez `.venv\Scripts\activate`

pip install --upgrade pip
pip install streamlit langchain llama-cpp-python huggingface_hub
```

Assurez‑vous d'avoir installé la version la plus récente de `llama-cpp-python` pour supporter les fichiers `.gguf`. Vous pouvez télécharger le modèle **Mistral-7B-Instruct-v0.1.Q4_0.gguf** depuis Hugging Face :

```bash
huggingface-cli download TheBloke/Mistral-7B-Instruct-v0.1-GGUF mistral-7b-instruct-v0.1.Q4_0.gguf --local-dir ./models
```


## 2. Aperçu de la solution

Le chatbot local doit remplir les fonctions suivantes :

- **Interface utilisateur :** Utiliser Streamlit pour créer une page web avec un titre, un champ de texte dans la barre latérale permettant de définir la personnalité (prompt système) du chatbot, puis des composants `st.chat_message`/`st.chat_input` pour afficher et saisir des messages.
- **Modèle quantifié :** Charger le modèle quantifié `.gguf` via `llama-cpp-python` et l'intégrer dans un wrapper `LlamaCpp` de LangChain.
- **Mémoire conversationnelle :** Utiliser `ConversationBufferMemory` de LangChain pour conserver un historique limité des messages afin que le modèle puisse répondre en contexte.
- **Chaîne LangChain :** Composer une `ChatPromptTemplate` avec un message système, l'historique et la nouvelle requête utilisateur, puis combiner le modèle et la mémoire via `RunnablePassthrough`/`RunnableLambda` pour obtenir la réponse.
- **Streaming des tokens :** Pour améliorer l'expérience, le wrapper `LlamaCpp` peut être initialisé avec `streaming=True` et une fonction de callback qui affiche les tokens au fur et à mesure.

Les cellules de code suivantes montrent une implémentation de référence que vous pouvez adapter et exécuter dans Colab ou en local. Les importations sont encapsulées dans un bloc `try/except` afin de ne pas provoquer d'erreur dans cet environnement.


In [ ]:
# Importations (encapsulées pour éviter une erreur si les packages ne sont pas installés)
try:
    from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
    from langchain.schema.output_parser import StrOutputParser
    from langchain.llms import LlamaCpp
    from langchain.memory import ConversationBufferMemory
    from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
except ImportError as e:
    print("Certaines librairies manquent :", e)

# Indiquez ici le chemin du modèle quantifié (.gguf)
MODEL_PATH = "./models/mistral-7b-instruct-v0.1.Q4_0.gguf"  # à adapter selon votre arborescence

# Personnalité par défaut (prompt système)
def get_default_personality():
    return "You are a helpful AI assistant."

# Fonction pour créer le LLM (wrapper LlamaCpp)
def create_llama_model(model_path: str, temperature: float = 0.7, max_tokens: int = 512, streaming: bool = True):
    """Initialise un modèle LlamaCpp depuis un fichier GGUF."""
    llm = LlamaCpp(
        model_path=model_path,
        n_ctx=4096,
        temperature=temperature,
        max_tokens=max_tokens,
        streaming=streaming,
    )
    return llm

# Fonction pour créer le prompt et la chaîne
def create_chat_chain(llm, system_prompt: str):
    """Construit le pipeline LangChain avec prompt et mémoire."""
    # Préparer le template du prompt
    system_message = SystemMessagePromptTemplate.from_template(system_prompt)
    history_placeholder = "{chat_history}"
    human_message = HumanMessagePromptTemplate.from_template("{user_input}")
    prompt = ChatPromptTemplate.from_messages([
        system_message,
        ("placeholder", history_placeholder),
        human_message,
    ])

    # Mémoire conversationnelle
    memory = ConversationBufferMemory(
        return_messages=True,
        memory_key="chat_history",
        input_key="user_input",
    )

    # Fonction qui invoque la chaîne
    def call_chain(inputs: dict):
        chain = prompt | llm | StrOutputParser()
        return chain.invoke(inputs)

    chain = RunnablePassthrough.assign(history=RunnableLambda(lambda x: memory.load_memory_variables({}))) | RunnableLambda(
        lambda inp: call_chain({
            "user_input": inp["user_input"],
            "chat_history": memory.chat_memory.messages
        })
    )
    return chain, memory

# Exemple d'utilisation (à tester dans un environnement où les dépendances sont installées)
"""
# llm = create_llama_model(MODEL_PATH)
# chain, memory = create_chat_chain(llm, get_default_personality())
# user_inputs = ["Hello, who are you?", "What is the capital of France?", "And who is the president?"]
# for message in user_inputs:
#     response = chain.invoke({"user_input": message})
#     memory.chat_memory.add_user_message(message)
#     memory.chat_memory.add_ai_message(response)
#     print(f"Assistant: {response}
")
"""


## 3. Interface utilisateur Streamlit

Créez un fichier `main.py` qui contient le code Streamlit pour démarrer l'application. Cette structure suit les étapes décrites dans l'exercice :

```python
import streamlit as st
from langchain.llms import LlamaCpp
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

MODEL_PATH = "./models/mistral-7b-instruct-v0.1.Q4_0.gguf"
llm = LlamaCpp(model_path=MODEL_PATH, n_ctx=4096, temperature=0.7, max_tokens=512, streaming=True)

# Initialiser l'état de session
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "system_prompt" not in st.session_state:
    st.session_state.system_prompt = "You are a helpful AI assistant."

# Fonction pour construire la chaîne
def build_chain(system_prompt: str):
    system_message = SystemMessagePromptTemplate.from_template(system_prompt)
    prompt = ChatPromptTemplate.from_messages([
        system_message,
        ("placeholder", "{chat_history}"),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ])
    memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history", input_key="user_input")
    def call_chain(inputs: dict):
        chain = prompt | llm | StrOutputParser()
        return chain.invoke(inputs)
    chain = RunnablePassthrough.assign(history=RunnableLambda(lambda x: memory.load_memory_variables({}))) | RunnableLambda(
        lambda inp: call_chain({"user_input": inp["user_input"], "chat_history": memory.chat_memory.messages})
    )
    return chain, memory

st.title("AI Chatbot with Memory")
st.sidebar.header("Personality Settings")
st.session_state.system_prompt = st.sidebar.text_area(
    "Enter system prompt for your assistant", st.session_state.system_prompt
)
chain, memory = build_chain(st.session_state.system_prompt)

# Afficher l'historique
for msg in st.session_state.chat_history:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("Ask something...")
if user_input:
    st.session_state.chat_history.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    response = chain.invoke({"user_input": user_input})
    st.session_state.chat_history.append({"role": "assistant", "content": response})
    with st.chat_message("assistant"):
        st.markdown(response)
```

Ce squelette montre comment gérer l'état conversationnel, personnaliser le prompt système et afficher les messages en continu.


## 4. Conclusion

Vous disposez maintenant d'un guide complet pour créer un chatbot local à mémoire courte reposant sur un modèle quantifié. Pour aller plus loin :

- Expérimentez différentes personnalités en modifiant le `system_prompt`.
- Ajustez les paramètres du modèle (`temperature`, `max_tokens`) pour varier la créativité des réponses.
- Implémentez le streaming de tokens pour afficher les réponses en temps réel dans l'interface Streamlit.
- Intégrez votre assistant dans d'autres projets (applications Flask, API FastAPI, etc.).

Partagez ensuite votre notebook (ou votre dépôt GitHub) et soumettez-le via la plateforme d'apprentissage.
